In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, GroupKFold

In [2]:
from missforest import MissForest
import warnings

In [ ]:


class MissForestImputerMerger(BaseEstimator, TransformerMixin):
    """
    Single transformer that:
    1. Extracts unique demographic rows (one per participant)
    2. Imputes using MissForest (handles mixed data types automatically)
    3. Merges with physiological data
    """
    
    def __init__(self, demographic_cols, categorical_cols,
                 participant_id_col='participant_name', 
                 max_iter=10):
        self.demographic_cols = demographic_cols
        self.categorical_cols = categorical_cols 
        self.participant_id_col = participant_id_col
        self.max_iter = max_iter
        
        # Will be fitted
        self.imputer_ = None
        
    def fit(self, X):
        """Fit MissForest on unique demographic rows"""
        # Extract unique demographic rows (one per participant)
        demo_unique = X.groupby(self.participant_id_col)[self.demographic_cols].first()
        
        # Identify categorical column indices
        """  cat_vars = None
        if self.categorical_cols:
            cat_vars = [demo_unique.columns.get_loc(col) 
                       for col in self.categorical_cols 
                       if col in demo_unique.columns]
            if not cat_vars:  # Empty list
                cat_vars = None """
        
        # Create and fit MissForest imputer
        self.imputer_ = MissForest (
            categorical=self.categorical_cols
        )
        
        # Fit on demographic data
        self.imputer_.fit(demo_unique)
        
        return self
    
    def transform(self, X):
        """Impute demographics and merge back"""
        
        # Extract unique demographic rows with participant IDs
        demo_unique = X.groupby(self.participant_id_col)[self.demographic_cols].first()
        
        # Impute
        demo_imputed = self.imputer_.transform(demo_unique)
        
        
        # Drop original demographic columns from X
        X_no_demo = X.drop(columns=self.demographic_cols)
        
        # Merge imputed demographics back
        X_merged = X_no_demo.merge(
            demo_imputed,
            left_on=self.participant_id_col,
            right_index=True,
            how='left'
        )
        
        
        return X_merged

In [ ]:

""" class ColumnDropper(BaseEstimator, TransformerMixin):
    #Drop specified columns
    
    def __init__(self, columns_to_drop):
        self.columns_to_drop = columns_to_drop
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return X.drop(columns=self.columns_to_drop, errors='ignore') """

""" 
class CategoricalEncoder(BaseEstimator, TransformerMixin):
    #One-hot encode categorical variables for modeling
    
    def __init__(self, categorical_cols):
        self.categorical_cols = categorical_cols
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        categorical_present = [c for c in self.categorical_cols if c in X.columns]
        
        if categorical_present:
            return pd.get_dummies(
                X, 
                columns=categorical_present, 
                drop_first=True,
                dtype=float
            )
        
        return X """




In [ ]:
pipeline = Pipeline([
        # Step 1: Impute and merge demographics using MissForest
        ('missforest_imputer_merger', MissForestImputerMerger(
            demographic_cols=demographic_cols,
            categorical_cols=categorical_cols,
            participant_id_col=participant_id_col,
            max_iter=10,
            random_state=42,
            n_estimators=100
        )),
        
        # Step 2: One-hot encode categorical variables for final model
        ('categorical_encoder', CategoricalEncoder(
            categorical_cols=categorical_cols
        )),
        
        # Step 3: Drop participant_id
        ('column_dropper', ColumnDropper(
            columns_to_drop=[participant_id_col]
        )),
        
        # Step 4: Scale features
        ('scaler', StandardScaler()),
        
        # Step 5: ElasticNet
        ('elasticnet', ElasticNet(max_iter=10000, random_state=42))
    ])

In [ ]:
outer_cv = GroupKFold(n_splits=5)
inner_cv = GroupKFold(n_splits=3)

test_rmse_scores = []
test_r2_scores = []

alphas = np.logspace(-4, 1, 10)
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]

param_grid = {
    'elasticnet__alpha': alphas,
    'elasticnet__l1_ratio': l1_ratios
}

running the pipeline below

In [57]:
PATH_TO_FINAL_DATASET = "data_cleaned/final_dataset.parquet"

In [58]:
df = pd.read_parquet(PATH_TO_FINAL_DATASET, engine='pyarrow')

In [59]:
df.columns

Index(['participant_name', 'timeInterval_ID', 'phq8_score',
       'sleep_missing_last14', 'sleep_min', 'sleep_max', 'sleep_mean',
       'sleep_median', 'sleep_std', 'sleep_iqr', 'age', 'gender_at_birth',
       'marital_status', 'children', 'ethnicity_1', 'eduction_finish_age',
       'drop_school_1', 'income_gbp', 'accommodation', 'mh_smoking', 'audit_1',
       'depression_medication'],
      dtype='str')

In [60]:
phq8_demographics = df.drop(columns=[col for col in df.columns if col.startswith('sleep_')])
phq8_demographics = phq8_demographics.drop(columns = ["timeInterval_ID"])

In [61]:
phq8_demographics.head(5)

,participant_name,phq8_score,age,gender_at_birth,marital_status,children,ethnicity_1,eduction_finish_age,drop_school_1,income_gbp,accommodation,mh_smoking,audit_1,depression_medication
0,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1,55.0,Female,Married and living with your husband/wife,Yes,English/Welsh/Scottish/Northern Irish/British,17.0,No,"More than £55,000",Own outright,"No, I have never smoked",4 or more times a week,NaN
1,9078ae18-c644-4cdb-a582-6a2aa671ecb4,1,55.0,Female,Married and living with your husband/wife,Yes,English/Welsh/Scottish/Northern Irish/British,17.0,No,"More than £55,000",Own outright,"No, I have never smoked",4 or more times a week,NaN
2,9078ae18-c644-4cdb-a582-6a2aa671ecb4,3,55.0,Female,Married and living with your husband/wife,Yes,English/Welsh/Scottish/Northern Irish/British,17.0,No,"More than £55,000",Own outright,"No, I have never smoked",4 or more times a week,NaN
3,9078ae18-c644-4cdb-a582-6a2aa671ecb4,11,55.0,Female,Married and living with your husband/wife,Yes,English/Welsh/Scottish/Northern Irish/British,17.0,No,"More than £55,000",Own outright,"No, I have never smoked",4 or more times a week,NaN
4,9078ae18-c644-4cdb-a582-6a2aa671ecb4,5,55.0,Female,Married and living with your husband/wife,Yes,English/Welsh/Scottish/Northern Irish/British,17.0,No,"More than £55,000",Own outright,"No, I have never smoked",4 or more times a week,NaN


In [78]:
X = phq8_demographics.drop(columns=['phq8_score'])
y = phq8_demographics['phq8_score']

In [98]:
demographic_cols = [
    'age',
    'gender_at_birth',
    'marital_status',
    'children',
    'ethnicity_1',
    'eduction_finish_age',
    'drop_school_1',
    'income_gbp',
    'accommodation',
    'mh_smoking',
    'audit_1',
    'depression_medication'
]

In [80]:
categorical_cols = ['marital_status', 'ethnicity_1', 'accommodation']


In [81]:
for col in categorical_cols:
    X[col] = X[col].cat.codes
    X[col] = X[col].replace(-1, np.nan)

In [82]:
for col in categorical_cols:
    print(f"{col}: {X[col].dtype}")

marital_status: float64
ethnicity_1: float64
accommodation: float64


In [83]:
ordinal_cols = ['income_gbp', 'mh_smoking', 'audit_1']

In [84]:
for col in ordinal_cols:
    print(f"{col}: {X[col].dtype}")

income_gbp: category
mh_smoking: category
audit_1: category


In [85]:
for col in ordinal_cols:
    X[col] = X[col].cat.codes
    X[col] = X[col].replace(-1, np.nan)

In [87]:
# Map boolean variables
X['gender_at_birth'] = X['gender_at_birth'].map({'Female': 0, 'Male': 1})
X['children'] = X['children'].map({'No': 0, 'Yes': 1})
X['drop_school_1'] = X['drop_school_1'].map({'No': 0, 'Yes': 1})
X['depression_medication'] = X['depression_medication'].map({'No': 0, 'Yes': 1})

In [95]:
X.columns

Index(['participant_name', 'age', 'gender_at_birth', 'marital_status',
       'children', 'ethnicity_1', 'eduction_finish_age', 'drop_school_1',
       'income_gbp', 'accommodation', 'mh_smoking', 'audit_1',
       'depression_medication'],
      dtype='str')

In [96]:
X.head()

,participant_name,age,gender_at_birth,marital_status,children,ethnicity_1,eduction_finish_age,drop_school_1,income_gbp,accommodation,mh_smoking,audit_1,depression_medication
0,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,0,2.0,1,0.0,17.0,0,4.0,0.0,0.0,4.0,NaN
1,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,0,2.0,1,0.0,17.0,0,4.0,0.0,0.0,4.0,NaN
2,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,0,2.0,1,0.0,17.0,0,4.0,0.0,0.0,4.0,NaN
3,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,0,2.0,1,0.0,17.0,0,4.0,0.0,0.0,4.0,NaN
4,9078ae18-c644-4cdb-a582-6a2aa671ecb4,55.0,0,2.0,1,0.0,17.0,0,4.0,0.0,0.0,4.0,NaN


In [99]:
imputer_merger = MissForestImputerMerger(
    demographic_cols=demographic_cols,
    categorical_cols=categorical_cols,
)

In [107]:
for cols in X.columns:
    print(f"{cols}: {X[cols].dtype}")

participant_name: str
age: float64
gender_at_birth: float64
marital_status: float64
children: float64
ethnicity_1: float64
eduction_finish_age: float64
drop_school_1: category
income_gbp: float64
accommodation: float64
mh_smoking: float64
audit_1: float64
depression_medication: float64


In [109]:
# cast gender_at_brith and depression_medication to float64
X['gender_at_birth'] = X['gender_at_birth'].astype(np.float64)  
X['depression_medication'] = X['depression_medication'].astype(np.float64)
X['children'] = X['children'].astype(np.float64)
X['drop_school_1'] = X['drop_school_1'].astype(np.float64)

In [111]:
X.groupby('participant_name')[demographic_cols].first()

,age,gender_at_birth,marital_status,children,ethnicity_1,eduction_finish_age,drop_school_1,income_gbp,accommodation,mh_smoking,audit_1,depression_medication
participant_name,,,,,,,,,,,,
007751c5-d7ad-4bec-a58f-abf32500e2ae,55.0,NaN,2.0,1.0,NaN,27.0,NaN,NaN,4.0,2.0,0.0,1.0
00bf567f-30ef-4317-9c3d-f62a37bcce27,56.0,0.0,1.0,1.0,17.0,21.0,0.0,NaN,2.0,1.0,1.0,1.0
0104dfff-4dcd-48ff-b912-51362f098ed0,34.0,0.0,1.0,1.0,17.0,21.0,0.0,NaN,1.0,1.0,1.0,NaN
01318193-bb33-4c1b-b1bf-d316a014b5ad,62.0,NaN,5.0,1.0,NaN,19.0,NaN,NaN,0.0,2.0,3.0,1.0
0157930f-10f7-4d5f-9575-a65efbd1d505,20.0,0.0,0.0,0.0,0.0,20.0,0.0,0.0,5.0,1.0,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
fa077fc5-d923-4bc1-bd1f-34e679af0025,29.0,0.0,0.0,0.0,4.0,26.0,0.0,3.0,5.0,1.0,4.0,1.0
fa4950dd-b952-40fa-a66f-4206cb108318,33.0,0.0,0.0,0.0,0.0,31.0,0.0,1.0,5.0,2.0,3.0,1.0
fb62039c-0b79-4be0-a7b9-ebf5d926eff2,53.0,NaN,2.0,1.0,NaN,12.0,NaN,NaN,3.0,2.0,0.0,1.0


In [110]:
imputer_merger.fit(X)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:333: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
100%|██████████| 5/5 [00:37<00:00,  7.58s/it]


,demographic_cols,"['age', 'gender_at_birth', ...]"
,categorical_cols,"['marital_status', 'ethnicity_1', ...]"
,participant_id_col,'participant_name'
,max_iter,10


In [112]:
X_imputed = imputer_merger.transform(X)

/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:490: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
/Users/kaylenemorrison/opt/anaconda3/envs/thesis/lib/python3.14/site-packages/missforest/missforest.py:494: UserWarning: In version 4.2.3, estimator fitting process is moved to `fit` method. `MissForest` will now imputes unseen missing values with fitted estimators with `transform` method. To retain the old behaviour, use `fit_transform` to fit the whole unseen data instead.
  warnings.warn(f"In version {VERSION}, estimator fitting process "
100%|██████████| 5/5 [00:00<00:00, 40.66it/s]


In [116]:
X.isnull().sum()

participant_name            0
age                         1
gender_at_birth          1837
marital_status              1
children                   15
ethnicity_1              1837
eduction_finish_age         3
drop_school_1            1837
income_gbp               3717
accommodation               1
mh_smoking                  1
audit_1                   186
depression_medication    1405
dtype: int64

In [115]:
# write code to sum the number of missing values in X
X_imputed.isnull().sum()

participant_name         0
age                      0
gender_at_birth          0
marital_status           0
children                 0
ethnicity_1              0
eduction_finish_age      0
drop_school_1            0
income_gbp               0
accommodation            0
mh_smoking               0
audit_1                  0
depression_medication    0
dtype: int64

In [114]:
# write code to check that for each participant, the demographic values are the same across all rows after imputation
demo_check = X_imputed.groupby('participant_name')[demographic_cols].nunique()
demo_check[demo_check > 1]

,age,gender_at_birth,marital_status,children,ethnicity_1,eduction_finish_age,drop_school_1,income_gbp,accommodation,mh_smoking,audit_1,depression_medication
participant_name,,,,,,,,,,,,
007751c5-d7ad-4bec-a58f-abf32500e2ae,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00bf567f-30ef-4317-9c3d-f62a37bcce27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0104dfff-4dcd-48ff-b912-51362f098ed0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
01318193-bb33-4c1b-b1bf-d316a014b5ad,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0157930f-10f7-4d5f-9575-a65efbd1d505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
fa077fc5-d923-4bc1-bd1f-34e679af0025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fa4950dd-b952-40fa-a66f-4206cb108318,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fb62039c-0b79-4be0-a7b9-ebf5d926eff2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [119]:
X[demographic_cols][X[demographic_cols].isna().all(axis=1)]

,age,gender_at_birth,marital_status,children,ethnicity_1,eduction_finish_age,drop_school_1,income_gbp,accommodation,mh_smoking,audit_1,depression_medication
3605,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [117]:
print(X_imputed[demographic_cols].dtypes)

age                      float64
gender_at_birth          float64
marital_status           float64
children                 float64
ethnicity_1              float64
eduction_finish_age      float64
drop_school_1            float64
income_gbp               float64
accommodation            float64
mh_smoking               float64
audit_1                  float64
depression_medication    float64
dtype: object


In [ ]:
# label encode categorical variables for MissForest

In [ ]:
# leave numerical variables as is
# change ordinal variables to their cat codes but leave as numeric for MissForest
# convert binary variables to 0/1 but leave as numeric for MissForest

In [ ]:

for train_idx, test_idx in outer_cv.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train = groups.iloc[train_idx]
    
    # Precompute inner CV splits respecting groups
    inner_cv_splits = list(inner_cv.split(X_train, y_train, groups_train))
    
    # Hyperparameter tuning with GridSearchCV on precomputed inner folds
    grid_search = GridSearchCV(pipeline, param_grid, cv=inner_cv_splits, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)  # Note: no pre-scaling, pipeline handles it
    
    # Best model
    best_model = grid_search.best_estimator_
    print(f"Best params for fold: {grid_search.best_params_}")
    
    # Evaluate on test set (pipeline includes scaler, so transform X_test)
    y_pred = best_model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    test_rmse_scores.append(rmse)
    test_r2_scores.append(r2)

# print all test rmse scores and r2 scores
print(f"Test RMSE scores: {test_rmse_scores}")
print(f"Test R² scores: {test_r2_scores}")

print(f"Test RMSE: {np.mean(test_rmse_scores):.4f} (+/- {np.std(test_rmse_scores):.4f})")
print(f"Test R²: {np.mean(test_r2_scores):.4f} (+/- {np.std(test_r2_scores):.4f})")

In [ ]:
""" def setup_pipeline(demographic_cols, physiological_cols, categorical_cols,
                   participant_id_col='participant_id'):
    
    #Pipeline with MissForest imputation
    
    
    pipeline = Pipeline([
        # Step 1: Impute and merge demographics using MissForest
        ('missforest_imputer_merger', MissForestImputerMerger(
            demographic_cols=demographic_cols,
            categorical_cols=categorical_cols,
            participant_id_col=participant_id_col,
            max_iter=10,
            random_state=42,
            n_estimators=100
        )),
        
        # Step 2: One-hot encode categorical variables for final model
        ('categorical_encoder', CategoricalEncoder(
            categorical_cols=categorical_cols
        )),
        
        # Step 3: Drop participant_id
        ('column_dropper', ColumnDropper(
            columns_to_drop=[participant_id_col]
        )),
        
        # Step 4: Scale features
        ('scaler', StandardScaler()),
        
        # Step 5: ElasticNet
        ('elasticnet', ElasticNet(max_iter=10000, random_state=42))
    ])
    
    return pipeline """


""" def run_grouped_cv(phys_data, demo_data, 
                   demographic_cols, physiological_cols, categorical_cols): """
    """
    Workflow with MissForest and grouped CV
    
    Parameters:
    -----------
    phys_data : pd.DataFrame
        Longitudinal physiological data with participant_id and phq8
    demo_data : pd.DataFrame
        Demographic data with participant_id and missing values
    demographic_cols : list
        All demographic column names
    physiological_cols : list
        Physiological feature column names
    categorical_cols : list
        Subset of demographic_cols that are categorical
    """
    
    """ # Merge datasets
    merged_data = phys_data.merge(demo_data, on='participant_id', how='left')
    
    # Prepare features and target
    feature_cols = physiological_cols + demographic_cols + ['participant_id']
    X = merged_data[feature_cols].copy()
    y = merged_data['phq8'].values
    groups = merged_data['participant_id'].values
    
    # Setup pipeline
    pipeline = setup_pipeline(
        demographic_cols=demographic_cols,
        physiological_cols=physiological_cols,
        categorical_cols=categorical_cols,
        participant_id_col='participant_id'
    )
    
    # Parameter grid
    param_grid = {
        'missforest_imputer_merger__max_iter': [5, 10],
        'missforest_imputer_merger__n_estimators': [50, 100],
        'elasticnet__alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
        'elasticnet__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
    }
    
    # GroupKFold CV
    group_cv = GroupKFold(n_splits=5)
    
    # GridSearchCV
    grid_search = GridSearchCV(
        pipeline,
        param_grid,
        cv=group_cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,  # MissForest handles parallelization internally
        verbose=2,
        error_score='raise'
    )
    
    # Fit
    print("Starting GridSearchCV with MissForest imputation...")
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore')
        grid_search.fit(X, y, groups=groups)
    
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best score (neg MSE): {grid_search.best_score_:.4f}")
    print(f"Best RMSE: {np.sqrt(-grid_search.best_score_):.4f}")
    
    return grid_search


# Example usage
if __name__ == "__main__":
    np.random.seed(42)
    
    # Sample physiological data (multiple rows per participant)
    n_participants = 50
    measurements_per_participant = 3
    
    phys_rows = []
    for pid in range(1, n_participants + 1):
        for _ in range(measurements_per_participant):
            phys_rows.append({
                'participant_id': pid,
                'heart_rate': np.random.normal(75, 10),
                'blood_pressure': np.random.normal(120, 15),
                'activity_level': np.random.uniform(0, 100),
                'phq8': np.random.poisson(8)
            })
    
    phys_data = pd.DataFrame(phys_rows)
    
    # Sample demographic data with missing values
    demo_rows = []
    for pid in range(1, n_participants + 1):
        demo_rows.append({
            'participant_id': pid,
            # Numerical
            'age': np.random.normal(45, 15) if np.random.rand() > 0.2 else np.nan,
            'bmi': np.random.normal(25, 5) if np.random.rand() > 0.15 else np.nan,
            # Categorical
            'sex': np.random.choice(['Male', 'Female']) if np.random.rand() > 0.1 else np.nan,
            'education': np.random.choice(['HS', 'BS', 'MS', 'PhD']) if np.random.rand() > 0.25 else np.nan,
            # Binary/Ordinal (MissForest handles these as numerical)
            'smoker': np.random.choice([0, 1]) if np.random.rand() > 0.15 else np.nan,
            'income_level': np.random.choice([1, 2, 3, 4, 5]) if np.random.rand() > 0.3 else np.nan,
        })
    
    demo_data = pd.DataFrame(demo_rows)
    
    print("Missing values in demographic data:")
    print(demo_data.isnull().sum())
    print(f"\nTotal missing: {demo_data.isnull().sum().sum()} out of {demo_data.size} values "
          f"({100*demo_data.isnull().sum().sum()/demo_data.size:.1f}%)\n")
    
    # Define columns
    demographic_cols = ['age', 'bmi', 'sex', 'education', 'smoker', 'income_level']
    physiological_cols = ['heart_rate', 'blood_pressure', 'activity_level']
    categorical_cols = ['sex', 'education']  # Only true categoricals
    
    # Run analysis
    results = run_grouped_cv(
        phys_data, demo_data,
        demographic_cols, physiological_cols, categorical_cols
    )
    
    print("\nTop 10 CV Results:")
    cv_results = pd.DataFrame(results.cv_results_)
    print(cv_results[['params', 'mean_test_score', 'std_test_score']]
          .sort_values('mean_test_score', ascending=False).head(10))
    
    # Access the best model
    best_model = results.best_estimator_
    
    # You can also look at feature coefficients
    feature_names = best_model.named_steps['scaler'].get_feature_names_out()
    coefficients = best_model.named_steps['elasticnet'].coef_
    
    print("\nTop 10 features by absolute coefficient:")
    coef_df = pd.DataFrame({
        'feature': feature_names,
        'coefficient': coefficients
    }).sort_values('coefficient', key=abs, ascending=False)
    print(coef_df.head(10)) """